In [0]:
from abc import ABC, abstractmethod
from lib.gateway.database import SparkSQLDatabaseGateway
from lib.interactor.governance import GovernanceInteractor
from lib.interactor.surrogate_key import SurrogateKeyInteractor

import pandas as pd
import yaml

from pyspark.sql import DataFrame
from delta.tables import DeltaTable

In [0]:
path_base = '/Workspace/Users/armando.n90@gmail.com/databricks_case/lakehouse/governance/metadata/assets/'
domain = 'medallion'
environment = 'dev'
source = 'healthsys'
plain_tables = ['cat_cie_10', 'certificates', 'claims', 'consultas', 'terms']
plain_tables = ['cat_cie_10']

database_gateway = SparkSQLDatabaseGateway()
governance_interactor = GovernanceInteractor(database_gateway=database_gateway)
surrogate_key_interactor = SurrogateKeyInteractor(database_gateway=database_gateway)

for plain_table in plain_tables:
    
    print('Processing plain table:', plain_table)
    path = f'{path_base}/{domain}/{source}/{plain_table}.yml'

    with open(path) as file:
        try:
            metadata = yaml.safe_load(file)
        except yaml.YAMLError as exc:
            print(exc)

    print(metadata)

    for layer in metadata['layers']:

        layer_name = layer['layer']
        print('Processing layer: ' + layer_name)

        catalog_name = f'{domain}_{environment}'
        table_name = layer_name +  '_' + metadata['name']
        print('Ingesting metadata table: ' + table_name)
        
        etl_module = None
        schema_name = None
        quality = None
        table_type = None

        if layer_name == 'raw':
            etl_module = metadata['name'] + '__load'
            schema_name = f'bronze_{source}'
            quality = 'bronze'
            table_type = 'table'
  
        catalog_id = governance_interactor.get_catalog_id(catalog_name)
        schema_id = governance_interactor.get_schema_id(catalog_id, schema_name)
    
        base_dict = {'schema_id': schema_id, 'table_name': table_name, 'version': 1}
        table_id = surrogate_key_interactor.assign_surrogate_key(catalog_name='governance_prod', schema_name='metadata', 
                                                            table_name='tables', base_values=base_dict, surrogate_column='table_id')

        write_mode = layer['write_mode']
        version = layer['version']
        current_flag = True
        valid_from = pd.Timestamp.utcnow().strftime('%Y-%m-%d %H:%M:%S') 
        valid_to = '2200-01-01 00:00:00'
        description = layer['description']
        owner = layer['owner']
        retention_policy = layer['retention_policy']

        columns = ['table_id', 'schema_id', 'table_name', 'etl_module', 'write_mode', 'quality', 
                'table_type', 'version', 'current_flag', 'valid_from', 'valid_to', 'description', 
                'owner', 'retention_policy']
        
        ingestion = [(table_id, schema_id, table_name, etl_module, write_mode, quality, 
                    table_type, version, current_flag, valid_from, valid_to, description, 
                    owner, retention_policy)]

        dataframe_ingestion = spark.createDataFrame(ingestion, columns)
        governance_interactor.merge_dataframe(dataframe_ingestion, catalog_name='governance_prod', 
                                            schema_name='metadata', table_name='tables', surrogate_column='table_id')

        array_data_columns = []
        i = 1
        for column in layer['schema']:

            column_name = column['column']
            base_dict = {'table_id': table_id, 'column_name': column_name}
            print('Ingesting metadata table details for column: ' + column_name)
        
            column_id = surrogate_key_interactor.assign_surrogate_key(catalog_name='governance_prod', 
                                                                schema_name='metadata', 
                                                                table_name='tables_detail', 
                                                                base_values=base_dict, 
                                                                surrogate_column='column_id')

            data_column = {}
            data_column['table_id'] = table_id
            data_column['column_id'] = column_id
            data_column['column_name'] = column_name
            data_column['data_type'] = column['data_type']
            data_column['ordinal_position'] = i
            data_column['is_nullable'] = column['is_nullable']
            data_column['is_partition'] = column['is_partition']
            data_column['comment'] = column['comment']
            array_data_columns.append(data_column)

            columns = ['table_id', 'column_id', 'column_name', 'data_type', 'ordinal_position', 
                    'is_nullable', 'is_partition', 'comment']
        
            ingestion = [(table_id, column_id, column_name, column['data_type'], i, 
                        column['is_nullable'], column['is_partition'], column['comment'])]

            dataframe_ingestion = spark.createDataFrame(ingestion, columns)

            governance_interactor.merge_dataframe(dataframe_ingestion, catalog_name='governance_prod', 
                                                schema_name='metadata', table_name='tables_detail', surrogate_column='column_id')

            i = i + 1
        
    print('')

In [0]:
test = spark.sql(f"select * from governance_prod.metadata.tables")
test.show(100)

In [0]:
test = spark.sql(f"select * from governance_prod.metadata.tables_detail")
test.show(100)